# *Clustering* con K-Means

## Introducción

***Clustering*** es una técnica de **aprendizaje no supervisado** que agrupa registros en grupos de características similares, conocidos como ***clusters***.

**K-Means** es un algoritmo de clustering que agrupa datos intentando separar muestras en $K$ grupos predefinidos, donde cada muestra pertenece al grupo cuya media (el **centroide** del cluster) es la más cercana.

Los grupos se establecen minimizando la **inercia o WCSS (Within-Cluster Sum of Squares)**, que es la suma de las distancias al cuadrado de cada muestra al centroide de su grupo.

- La **inercia** es una medida de qué tan coherentes son internamente los clusters. En el contexto de K-Means, también se llama **Within-Cluster Sum of Squares (WCSS)**.
- Para cada cluster, el WCSS es la suma de las distancias al cuadrado entre cada punto del cluster y el centroide de ese cluster.
- La inercia total es la suma del WCSS de todos los clusters:

$$\text{Inercia} = \sum_{i=1}^K \sum_{x \in C_i} \|x - \mu_i\|^2$$

donde $C_i$ es el conjunto de puntos en el cluster $i$ y $\mu_i$ es el centroide del cluster $i$.

- Una inercia más baja significa que los puntos están más cerca de sus centroides, lo que indica clusters más compactos.

El algoritmo K-Means sigue estos pasos:
1. **Inicialización**: Elegir $K$ centroides iniciales de forma aleatoria o usando algún método específico.
2. **Asignación**: Asignar cada muestra al centroide más cercano.
3. **Actualización**: Recalcular los centroides como la media de las muestras en cada grupo.
4. **Iteración**: Repetir los pasos 2 y 3 hasta que los centroides ya no cambien (el algoritmo converge) o se alcance un número máximo de iteraciones.

[![K-means](../img/K-means_convergence.gif)](https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif)

## Creación de Datos de Prueba

Utilizaremos la función `make_blobs` de `sklearn.datasets` para generar datos de prueba. Esta función genera un dataset de puntos distribuidos como nubes de puntos. Pasamos los siguientes parámetros:
- `n_samples`: número total de muestras
- `n_features`: número de features para cada muestra. Trabajaremos solo con 2 features para poder visualizar los datos en 2D.
- `centers`: número de clusters que queremos generar
- `random_state`: semilla para la generación de números aleatorios
- `cluster_std`: desviación estándar de los clusters

Es importante recordar que estamos haciendo este proceso de representación para ayudar a entender el algoritmo, pero normalmente trabajaremos con datasets de muchas más dimensiones donde no podemos visualizar los datos. De hecho, en este caso normalmente sería fácil dividir en clusters a ojo. Y efectivamente, ya estamos generando datos de prueba con esta función `make_blobs` al indicar el número de clusters que queremos que genere.

In [ ]:
from sklearn import datasets
import matplotlib.pyplot as plt

X, y = datasets.make_blobs(n_samples=1000, # Número de muestras
                           n_features=2, # Número de features
                           centers=5, # Número de clusters
                           cluster_std=[0.5, 0.6, 0.8, 1, 1.1], # Desviación estándar de cada cluster
                           random_state=42,) # Semilla para reproducibilidad

# Estirar un eje. Esto hace que el escalado de features sea crucial para K-Means
X[:, 1] = X[:, 1] * 10 

plt.scatter(X[:, 0], X[:, 1], s=2)
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

### Escalado de Features


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(X[:, 0], X[:, 1], s=2)
plt.xlabel('x1')
plt.ylabel('x2')
plt.axis('equal')  # <--- Fuerza escala igual
plt.show()


Antes de aplicar K-Means, es **CRÍTICO** escalar los datos (por ejemplo, usando `StandardScaler`).

Dado que K-Means utiliza la distancia Euclidiana para asignar puntos a clusters, los features con escalas más grandes influirán desproporcionadamente en el resultado. Si una variable se mide en miles y otra en decimales, el algoritmo agrupará casi por completo en función de la variable más grande, dando resultados incorrectos.

Estandarizar los features garantiza que contribuyan equitativamente a los cálculos de distancia.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Visualizar datos escalados
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], s=2)
plt.xlabel('x1 (escalado)')
plt.ylabel('x2 (escalado)')
plt.show()

## Entrenamiento de K-Means

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5, # Número de clusters (valor de k)
    n_init='auto',
    random_state=0) # Número de veces que el algoritmo se ejecuta con diferentes centroides iniciales
y_pred = kmeans.fit_predict(X_scaled) # Equivalente a kmeans.fit(X_scaled) y luego y_pred = kmeans.predict(X_scaled)

El constructor de KMeans recibe los siguientes parámetros:
* `n_clusters`: El número de clusters a formar y el número de centroides a generar. Es lo que variamos en un bucle para encontrar el número óptimo de clusters.
* `init`: Método para inicializar los centroides. El más simple es 'random', pero por defecto usa 'k-means++', que es un método más sofisticado que intenta colocar los centroides iniciales lejos entre sí.
* `max_iter`: Número máximo de iteraciones del algoritmo para una sola ejecución.
* `n_init`: Número de veces que se ejecutará el algoritmo con diferentes centroides. El resultado final será la mejor salida de n_init ejecuciones consecutivas en términos de inercia.

Al igual que otros estimadores de `scikit-learn`, KMeans tiene un método `fit_predict` que puede usarse para entrenar el modelo y predecir los clusters a los que pertenecen los datos. Este método devuelve un array de **etiquetas de cluster**, que asigna cada punto de datos a un cluster, y también lo almacena en la propiedad `labels_` del objeto KMeans.

Con el operador `is` podemos verificar si dos variables apuntan a la misma dirección de memoria, es decir, si son el mismo objeto. En este caso, verificamos si el atributo `labels_` del objeto KMeans es el mismo que el array de etiquetas devuelto por el método `fit_predict`:

In [ ]:
y_pred is kmeans.labels_ 

True

Los centroides del cluster se pueden obtener con el atributo `cluster_centers_`.

In [ ]:
import matplotlib.pyplot as plt

# Transformación inversa de los centroides para que coincidan con la escala original de los datos
centroids = scaler.inverse_transform(kmeans.cluster_centers_)
print(centroids)

plt.scatter(X[:, 0], X[:, 1], s=2)
plt.scatter(centroids[:,0], centroids[:,1], c='r', s=30)

for i, center in enumerate(centroids):
    plt.annotate(i, center, fontsize=18)

plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

Podemos predecir fácilmente el cluster al que pertenece cada nueva instancia con el método `predict`:

In [ ]:
import numpy as np
X_new = np.array([[0, 12], [2.5, -12], [-3, 13], [-3, -15], [-3, 50]])

X_new_scaled = scaler.transform(X_new)
y_new = kmeans.predict(X_new_scaled)


plt.scatter(X[:, 0], X[:, 1], s=2)
plt.scatter(centroids[:,0], centroids[:,1], c='r', s=20)
plt.scatter(X_new[:, 0], X_new[:, 1], c='g', s=50)

for i, center in enumerate(centroids):
    plt.annotate(i, center, fontsize=18)
    
for i, point in enumerate(X_new): # Anotación para los nuevos puntos
    plt.annotate(f'x{i}={y_new[i]}', point, fontsize=12)

plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

Al graficar las fronteras de decisión del algoritmo, obtenemos una [teselación de Voronoi](https://en.wikipedia.org/wiki/Voronoi_diagram):

In [ ]:
def plot_decision_regions(kmeans, X):
  x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1 # Valores mínimo y máximo para x1
  y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1 # Valores mínimo y máximo para x2
  xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), 
                      np.arange(y_min, y_max, 0.1)) 
  
  Z = kmeans.predict(np.c_[xx.ravel(), yy.ravel()])
  Z = Z.reshape(xx.shape)

  plt.figure(figsize=(8, 8))
  plt.contourf(xx, yy, Z, alpha=0.4)
  plt.scatter(X[:, 0], X[:, 1], c=y_pred, s=20, edgecolor='k')
  plt.show()
  
plot_decision_regions(kmeans,X_scaled)

La gran mayoría de las instancias fueron claramente asignadas a su cluster original.

Lo único que le importa a K-Means es la distancia entre las instancias y los centroides. En lugar de asignar cada instancia a un cluster (hard clustering), es mejor dar una puntuación de cluster por instancia (soft clustering). La puntuación puede ser la distancia entre la instancia y los centroides.

El método `transform` mide la distancia entre cada instancia y los centroides.

In [ ]:
kmeans.transform(X_new)

array([[10.96078819, 13.80286573, 12.27911435, 11.32035354, 11.81604276],
       [13.28496709, 10.76351827, 11.85106816, 13.29795998, 12.32750128],
       [12.32079492, 14.92932588, 13.8995097 , 12.3696834 , 13.34579102],
       [16.30958562, 13.39556631, 15.41554209, 15.84584225, 15.68452978],
       [49.04991394, 51.81980656, 50.3931413 , 49.2788106 , 49.9338825 ]])

## Determinar el Número Óptimo de Clusters

Dado que K-Means requiere que especifiquemos $K$ (número de clusters) de antemano, necesitamos métodos para encontrar el mejor $K$. Dos métodos comunes son el **Método del Codo** y el **Silhouette Score**.

### 1. El Método del Codo

El método del codo observa la **inercia** (WCSS) a medida que $K$ aumenta. La inercia siempre disminuye conforme $K$ aumenta, pero buscamos el punto "codo" donde la tasa de disminución se ralentiza drásticamente.

- A medida que $K$ aumenta, la inercia siempre disminuye (o permanece igual), porque agregar más clusters reduce la distancia promedio de los puntos a su centroide más cercano.
- Sin embargo, demasiados clusters pueden llevar a overfitting, por lo que buscamos el punto donde agregar más clusters no reduce significativamente la inercia: el punto "codo".

In [ ]:
inertia = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init='auto', random_state=42)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo')

El problema de este método es que a veces no hay un codo claro. Aquí podríamos considerar que el número óptimo de clusters es 3 o 4. Esto se debe a que aunque generamos 5 clusters (centers=5), el posicionamiento aleatorio (random_state=42) y las desviaciones estándar probablemente causaron que dos de los clusters se superpusieran significativamente.

### 2. Silhouette Score

El **Silhouette Score** mide qué tan similar es un objeto a su propio cluster (cohesión) en comparación con otros clusters (separación). El silhouette va de -1 a +1, donde un valor alto indica que el objeto está bien emparejado con su propio cluster y mal emparejado con los clusters vecinos.

* **+1**: La muestra está lejos de los clusters vecinos.
* **0**: La muestra está en o muy cerca de la frontera de decisión entre dos clusters vecinos.
* **-1**: La muestra puede haber sido asignada al cluster incorrecto.

In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
k_range_sil = range(2, 11) # Silhouette requiere al menos 2 clusters

for k in k_range_sil:
    kmeans = KMeans(n_clusters=k, n_init='auto', random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.figure(figsize=(8, 5))
plt.plot(k_range_sil, silhouette_scores, marker='o', color='green')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs K')
plt.show()

Podemos confirmar que el mayor silhouette score se obtiene al usar 4 clusters, y de hecho, $K$=3 es mayor que $K$=5.

## Limitaciones de K-Means

- `K-Means` no es perfecto, por lo que es necesario ejecutar el algoritmo varias veces para evitar soluciones subóptimas.
- Otro factor limitante del algoritmo es que necesitamos especificar el número de clusters.
- `K-Means` tampoco funciona muy bien cuando los grupos tienen tamaños diferentes, densidades diferentes o formas no esféricas.
- Dependiendo de los datos, diferentes algoritmos de clustering pueden funcionar mejor (como `DBSCAN` o `Gaussian Mixtures`).
- Escalar las entradas con StandardScaler también es esencial con `K-Means`.

## Recursos Adicionales

- https://github.com/tirthajyoti/Machine-Learning-with-Python/blob/master/Clustering-Dimensionality-Reduction/K_Means_Clustering_Practice.ipynb
- https://scikit-learn.org/stable/modules/clustering.html#k-means
- https://www.datacamp.com/tutorial/k-means-clustering-python
- [StatQuest: K-means clustering](https://www.youtube.com/watch?v=4b5d3muPQmA)